# Part 1: Acne Detection — Faster R-CNN

Training Faster R-CNN (ResNet-50 FPN backbone) on ACNE04 for multi-class acne lesion detection.
Selected as the second architecture based on YOLOv5 baseline results showing weakness on small dense lesions and poor localization (mAP@0.5:0.95 = 0.057).
Run this notebook on a **Colab GPU runtime (T4 or better)**.

## Contents
1. Environment setup
2. Data download (COCO format)
3. Verify environment
4. Model definition (Dataset, Faster R-CNN, training loop)
5. Train Faster R-CNN
6. Download results

## 1) Environment Setup
Clones the repo and installs dependencies including pycocotools.
**Restart the runtime after this cell completes before continuing.**

In [ ]:
!git clone https://github.com/kaizar-rang/yanglab-acne.git
%cd yanglab-acne
%pip install roboflow pycocotools -q
%pip install -r requirements.txt -q
print("\nDone. Restart the runtime (Runtime -> Restart session), then continue from Cell 2.")

## 2) Credentials and Data Download
Enter your Roboflow API key when prompted. Downloads ACNE04 in COCO format — required for Faster R-CNN's annotation structure (absolute pixel coordinates in a single JSON file vs YOLOv5's per-image .txt files).

In [ ]:
import numpy as np
from pathlib import Path
from PIL import Image # To load images faster since R-CNN expects PIL images (not openCV arrays)
from torch.utils.data import Dataset, DataLoader # Feed data to the model in batches
from torchvision.models.detection import fasterrcnn_resnet50_fpn # Pretrained model
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor # Used to output our 4 classes instead of 91 defuult COCO classes

In [ ]:
import os, getpass, json
from pathlib import Path

%cd /content/yanglab-acne
os.environ["WANDB_DISABLED"] = "true"

roboflow_key = getpass.getpass("Enter ROBOFLOW_API_KEY: ")
os.environ["ROBOFLOW_API_KEY"] = roboflow_key

from roboflow import Roboflow
rf = Roboflow(api_key=os.environ["ROBOFLOW_API_KEY"])
project = rf.workspace("acne-vulgaris-detection").project("acne04-detection")
version = project.version(1)
version.download("coco", location="data/acne04_coco", overwrite=True)

print(f"Setup complete. Working directory: {os.getcwd()}")

## 3) Verify Environment
Confirms GPU availability and PyTorch version before training.

In [ ]:
import torch
import torchvision
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

## 4) Model Definition
Defines:
- `AcneCocoDataset` — custom PyTorch Dataset that reads COCO JSON annotations
- `build_model` — loads pretrained Faster R-CNN and replaces the classification head for 4 classes
- `train_one_epoch` — single epoch training loop
- `evaluate` — validation loss loop
- `main` — full training orchestration with DataLoaders, optimizer, and scheduler

In [ ]:
# Tell PyTorch how to load one image from the dataset
class AcneCocoDataset(Dataset):
    def __init__(self, img_dir, annotation_file, transforms=None):
        self.img_dir = Path(img_dir)
        self.transforms = transforms
        
        # Load COCO annotations
        with open(annotation_file) as f:
            coco = json.load(f)
        
        # Build a mapping from image_id to annotations
        self.images = coco["images"]
        self.img_id_to_anns = {}
        for ann in coco["annotations"]:
            img_id = ann["image_id"]
            if img_id not in self.img_id_to_anns:
                self.img_id_to_anns[img_id] = []
            self.img_id_to_anns[img_id].append(ann)
    
    def __len__(self):
        return len(self.images)
    
    def __getitem__(self, idx):
        # Load image
        img_info = self.images[idx]
        img_path = self.img_dir / img_info["file_name"]
        image = Image.open(img_path).convert("RGB")
        
        # Get annotations for this image
        img_id = img_info["id"]
        anns = self.img_id_to_anns.get(img_id, [])
        
        # Build target dict that Faster R-CNN expects
        boxes = []
        labels = []
        for ann in anns:
            x, y, w, h = ann["bbox"]
            boxes.append([x, y, x + w, y + h])
            labels.append(ann["category_id"])
        
        target = {
            "boxes": torch.tensor(boxes, dtype=torch.float32),
            "labels": torch.tensor(labels, dtype=torch.int64),
            "image_id": torch.tensor([img_id])
        }
        
        # Handle images with no annotations
        if len(boxes) == 0:
            target["boxes"] = torch.zeros((0, 4), dtype=torch.float32)
            target["labels"] = torch.zeros(0, dtype=torch.int64)
        
        # Convert image to tensor
        image = torchvision.transforms.functional.to_tensor(image)
        
        return image, target

In [ ]:
def build_model(num_classes):
    # Load pretrained Faster R-CNN with ResNet-50 backbone
    model = fasterrcnn_resnet50_fpn(weights="DEFAULT")
    
    # Replace the classifier head to match our number of classes
    # +1 for background class which Faster R-CNN always includes
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes + 1)
    
    return model

In [ ]:
def train_one_epoch(model, optimizer, dataloader, device):
    model.train()
    total_loss = 0
    
    for images, targets in dataloader:
        # Move data to GPU
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
        
        # Forward pass — Faster R-CNN returns a dict of losses directly
        loss_dict = model(images, targets)
        losses = sum(loss for loss in loss_dict.values())
        
        # Backward pass
        optimizer.zero_grad()
        losses.backward()
        optimizer.step()
        
        total_loss += losses.item()
    
    return total_loss / len(dataloader)

In [ ]:
def evaluate(model, dataloader, device):
    model.eval()
    total_loss = 0
    
    with torch.no_grad():
        for images, targets in dataloader:
            images = [img.to(device) for img in images]
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
            
            # In eval mode with targets, Faster R-CNN still returns losses
            with torch.inference_mode(False):
                loss_dict = model(images, targets)
            losses = sum(loss for loss in loss_dict.values())
            total_loss += losses.item()
    
    return total_loss / len(dataloader)

## 5) Train Faster R-CNN
Trains for 20 epochs with SGD optimizer (lr=0.005, momentum=0.9) and StepLR scheduler.
Uses batch size 4 due to higher memory requirements vs YOLOv5.
Expected time: ~60-90 minutes on T4 GPU.

In [ ]:
def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")
    
    # Paths
    repo_root = Path("/content/yanglab-acne")
    train_img_dir = repo_root / "data/acne04_coco/train"
    val_img_dir = repo_root / "data/acne04_coco/valid"
    train_ann = repo_root / "data/acne04_coco/train/_annotations.coco.json"
    val_ann = repo_root / "data/acne04_coco/valid/_annotations.coco.json"
    
    # Datasets
    train_dataset = AcneCocoDataset(train_img_dir, train_ann)
    val_dataset = AcneCocoDataset(val_img_dir, val_ann)
    
    # DataLoaders
    train_loader = DataLoader(
        train_dataset, batch_size=4, shuffle=True,
        num_workers=2, collate_fn=lambda x: tuple(zip(*x))
    )
    val_loader = DataLoader(
        val_dataset, batch_size=4, shuffle=False,
        num_workers=2, collate_fn=lambda x: tuple(zip(*x))
    )
    
    # Model
    model = build_model(num_classes=4)
    model.to(device)
    
    # Optimizer
    optimizer = torch.optim.SGD(
        model.parameters(), lr=0.005, momentum=0.9, weight_decay=0.0005
    )
    
    # Learning rate scheduler
    lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=3, gamma=0.1)
    
    # Training loop
    num_epochs = 20
    for epoch in range(num_epochs):
        train_loss = train_one_epoch(model, optimizer, train_loader, device)
        val_loss = evaluate(model, val_loader, device)
        lr_scheduler.step()
        
        print(f"Epoch {epoch+1}/{num_epochs} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")
    
    # Save model
    os.makedirs("outputs/checkpoints/frcnn_acne", exist_ok=True)
    torch.save(model.state_dict(), "outputs/checkpoints/frcnn_acne/best.pt")
    print("Model saved to outputs/checkpoints/frcnn_acne/best.pt")

if __name__ == "__main__":
    main()

## 6. Download Results
**Run immediately after training finishes. Do not close the session first.**
Downloads model weights to your local machine.

In [ ]:
from google.colab import files

# Save and download model weights and results
os.makedirs("outputs/checkpoints/frcnn_acne", exist_ok=True)

!zip -r frcnn_results.zip outputs/checkpoints/frcnn_acne/
files.download('frcnn_results.zip')

print("Download initiated. Check your browser's download folder.")